In [ ]:
from pathlib import Path
from IPython.display import display
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from xgboost import XGBRFClassifier

from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedKFold,
    GridSearchCV,
)

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

MATRIX_PATH = Path(
    "related_files/ml_55_matrix_main_log1p_cleaned.tsv"
)

OUTPUT_DIR = Path(
    "results/ml_output_main_xgb"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 2026

N_SPLITS = 5
N_REPEATS = 20

GRID_N_JOBS = 5
XGB_N_JOBS = 5
RUN_SHAP = True

BLUE = "#0072B2"
ORANGE = "#D55E00"
GREEN = "#009E73"
GRAY = "#666666"

sns.set_theme(
    context="paper",
    style="ticks",
    font_scale=1.25
)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial"],
    "svg.fonttype": "none",
    "axes.linewidth": 1.0,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def save_figure(fig, output_dir, filename):
    output_dir = Path(output_dir)
    fig.savefig(output_dir / f"{filename}.svg", bbox_inches="tight", facecolor="white")

    fig.savefig(
        output_dir / f"{filename}.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        output_dir / f"{filename}.pdf",
        bbox_inches="tight",
        facecolor="white"
    )

df = pd.read_csv(MATRIX_PATH, sep="\t")
if len(df) != 55:
    raise ValueError(f"Expected 55 main-analysis species; found {len(df)}.")

required_columns = {
    "species_name",
    "main_label"
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Input file is missing columns: {missing_columns}"
    )

if df["species_name"].duplicated().any():
    duplicated_species = df.loc[
        df["species_name"].duplicated(),
        "species_name"
    ].tolist()

    raise ValueError(
        "Duplicate species in input: "
        + ", ".join(duplicated_species)
    )

feature_names = [
    column for column in df.columns
    if column not in [
        "species_name",
        "main_label"
    ]
]

feature_df = df[feature_names].apply(
    pd.to_numeric,
    errors="coerce"
)

all_missing_features = feature_df.columns[
    feature_df.isna().all()
].tolist()

if all_missing_features:
    raise ValueError(
        "Nonnumeric feature columns: \n"
        + "\n".join(all_missing_features)
    )

feature_df = feature_df.fillna(0)

X = feature_df.to_numpy(dtype=np.float32)

y = pd.to_numeric(
    df["main_label"],
    errors="raise"
).astype(int).to_numpy()

species = df[
    "species_name"
].astype(str).to_numpy()

if set(np.unique(y)) != {0, 1}:
    raise ValueError(
        "main_label must contain binary labels 0 and 1; "
        f"found: {np.unique(y)}"
    )

class_counts = (
    pd.Series(y)
    .value_counts()
    .sort_index()
)

if class_counts.min() < N_SPLITS:
    raise ValueError(
        f"Smallest class has {class_counts.min()} samples; "
        f"not enough for {N_SPLITS}-fold stratified cross-validation."
    )

print(f"Species: {len(y)}")
print(f"OG features: {len(feature_names)}")
print("\n class counts:")
print(class_counts)


In [ ]:
PARAM_GRID = {
    "n_estimators": [300, 500],
    "max_depth": [2, 3],
    "min_child_weight": [1, 2],
    "subsample": [0.6, 0.8],
    "colsample_bynode": [0.3, 0.6],
}

def make_binary_xgbrf(
    scale_pos_weight,
    random_state
):
    return XGBRFClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        importance_type="gain",
        scale_pos_weight=scale_pos_weight,
        n_jobs=XGB_N_JOBS,
        random_state=random_state,
        verbosity=0,
    )

def calculate_mean_abs_shap(model, X_test):
    explainer = shap.TreeExplainer(model)

    try:
        shap_values = explainer.shap_values(
            X_test,
            check_additivity=False
        )
    except TypeError:
        shap_values = explainer.shap_values(
            X_test
        )

    if isinstance(shap_values, list):
        shap_array = np.stack(
            shap_values,
            axis=-1
        )
    else:
        shap_array = np.asarray(
            shap_values
        )

    if shap_array.ndim == 2:
        return np.abs(shap_array).mean(axis=0)

    if shap_array.ndim == 3:
        if shap_array.shape[1] == X_test.shape[1]:
            return np.abs(shap_array).mean(
                axis=(0, 2)
            )

        if shap_array.shape[2] == X_test.shape[1]:
            return np.abs(shap_array).mean(
                axis=(0, 1)
            )

    raise ValueError(
        "Unsupported SHAP array shape: "
        f"{shap_array.shape}"
    )


In [ ]:
# Tune within each outer training fold; evaluate only on its held-out species.
outer_cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)

fold_results = []
prediction_tables = []
gain_tables = []
shap_tables = []

roc_curves = []
pr_curves = []

for fold_id, (train_idx, test_idx) in enumerate(
    outer_cv.split(X, y),
    start=1
):
    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    negative_count = np.sum(y_train == 0)
    positive_count = np.sum(y_train == 1)

    scale_pos_weight = (
        negative_count / positive_count
    )

    inner_cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=RANDOM_STATE + fold_id
    )

    search = GridSearchCV(
        estimator=make_binary_xgbrf(
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE + fold_id
        ),
        param_grid=PARAM_GRID,
        scoring="average_precision",
        cv=inner_cv,
        n_jobs=GRID_N_JOBS,
        pre_dispatch=GRID_N_JOBS,
        refit=True,
        return_train_score=False,
        error_score="raise",
    )

    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    test_probabilities = best_model.predict_proba(
        X_test
    )[:, 1]

    test_predictions = (
        test_probabilities >= 0.5
    ).astype(int)

    roc_auc = roc_auc_score(
        y_test,
        test_probabilities
    )

    pr_auc = average_precision_score(
        y_test,
        test_probabilities
    )

    balanced_accuracy = balanced_accuracy_score(
        y_test,
        test_predictions
    )

    accuracy = accuracy_score(
        y_test,
        test_predictions
    )

    f1 = f1_score(
        y_test,
        test_predictions,
        zero_division=0
    )

    precision = precision_score(
        y_test,
        test_predictions,
        zero_division=0
    )

    sensitivity = recall_score(
        y_test,
        test_predictions,
        pos_label=1,
        zero_division=0
    )

    specificity = recall_score(
        y_test,
        test_predictions,
        pos_label=0,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        test_predictions
    )

    fold_results.append({
        "fold": fold_id,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "balanced_accuracy": balanced_accuracy,
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "mcc": mcc,
        "best_inner_pr_auc": search.best_score_,
        "scale_pos_weight": scale_pos_weight,
        "best_params": json.dumps(
            search.best_params_,
            ensure_ascii=False
        ),
    })

    prediction_tables.append(
        pd.DataFrame({
            "species_name": species[test_idx],
            "fold": fold_id,
            "true_label": y_test,
            "predicted_label": test_predictions,
            "probability_class_1": test_probabilities,
        })
    )

    gain_tables.append(
        pd.Series(
            best_model.feature_importances_,
            index=feature_names,
            name=f"fold_{fold_id}"
        )
    )

    if RUN_SHAP:
        mean_abs_shap = calculate_mean_abs_shap(
            best_model,
            X_test
        )

        shap_tables.append(
            pd.Series(
                mean_abs_shap,
                index=feature_names,
                name=f"fold_{fold_id}"
            )
        )

    fpr, tpr, _ = roc_curve(
        y_test,
        test_probabilities
    )

    precision_curve, recall_curve, _ = (
        precision_recall_curve(
            y_test,
            test_probabilities
        )
    )

    roc_curves.append((fpr, tpr))

    pr_curves.append(
        (precision_curve, recall_curve)
    )

    print(
        f"Fold {fold_id:>3} | "
        f"ROC-AUC={roc_auc:.3f} | "
        f"PR-AUC={pr_auc:.3f} | "
        f"Balanced accuracy="
        f"{balanced_accuracy:.3f}"
    )

fold_df = pd.DataFrame(
    fold_results
)

predictions_df = pd.concat(
    prediction_tables,
    ignore_index=True
)

gain_df = pd.concat(
    gain_tables,
    axis=1
)

if RUN_SHAP:
    shap_df = pd.concat(
        shap_tables,
        axis=1
    )
else:
    shap_df = None


In [ ]:
metric_columns = [
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
    "accuracy",
    "f1",
    "precision",
    "sensitivity",
    "specificity",
    "mcc",
]

performance_summary = (
    fold_df[metric_columns]
    .agg(["mean", "std", "median", "min", "max"])
    .T
)

fold_df.to_csv(
    OUTPUT_DIR /
    "main_binary_fold_metrics.tsv",
    sep="\t",
    index=False
)

performance_summary.to_csv(
    OUTPUT_DIR /
    "main_binary_performance_summary.tsv",
    sep="\t"
)

predictions_df.to_csv(
    OUTPUT_DIR /
    "main_binary_out_of_fold_predictions.tsv",
    sep="\t",
    index=False
)

display(performance_summary)

species_prediction_summary = (
    predictions_df
    .groupby([
        "species_name",
        "true_label"
    ])
    .agg(
        mean_probability=(
            "probability_class_1",
            "mean"
        ),
        sd_probability=(
            "probability_class_1",
            "std"
        ),
        predicted_positive_fraction=(
            "predicted_label",
            "mean"
        ),
        n_test_predictions=(
            "fold",
            "size"
        ),
    )
    .reset_index()
)

species_prediction_summary.to_csv(
    OUTPUT_DIR /
    "main_binary_species_prediction_summary.tsv",
    sep="\t",
    index=False
)

fpr_grid = np.linspace(0, 1, 300)
recall_grid = np.linspace(0, 1, 300)

interpolated_tpr = []

for fpr, tpr in roc_curves:
    fold_tpr = np.interp(
        fpr_grid,
        fpr,
        tpr
    )
    fold_tpr[0] = 0
    interpolated_tpr.append(fold_tpr)

interpolated_tpr = np.vstack(
    interpolated_tpr
)

mean_tpr = interpolated_tpr.mean(axis=0)
std_tpr = interpolated_tpr.std(axis=0)
mean_tpr[-1] = 1

interpolated_precision = []

for precision_curve, recall_curve in pr_curves:
    fold_precision = np.interp(
        recall_grid,
        recall_curve[::-1],
        precision_curve[::-1]
    )
    interpolated_precision.append(
        fold_precision
    )

interpolated_precision = np.vstack(
    interpolated_precision
)

mean_precision = interpolated_precision.mean(
    axis=0
)

std_precision = interpolated_precision.std(
    axis=0
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10.4, 4.5)
)

axes[0].plot(
    fpr_grid,
    mean_tpr,
    color=BLUE,
    linewidth=2.3,
    label=(
        f"Mean ROC-AUC = "
        f"{fold_df['roc_auc'].mean():.3f}"
    )
)

axes[0].fill_between(
    fpr_grid,
    np.maximum(mean_tpr - std_tpr, 0),
    np.minimum(mean_tpr + std_tpr, 1),
    color=BLUE,
    alpha=0.16,
    linewidth=0,
    label="±1 SD across outer folds"
)

axes[0].plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color=GRAY,
    linewidth=1
)

axes[0].set(
    xlabel="False positive rate",
    ylabel="True positive rate",
    title="a  ROC curve"
)

axes[0].legend(
    loc="lower right"
)

axes[1].plot(
    recall_grid,
    mean_precision,
    color=ORANGE,
    linewidth=2.3,
    label=(
        f"Mean PR-AUC = "
        f"{fold_df['pr_auc'].mean():.3f}"
    )
)

axes[1].fill_between(
    recall_grid,
    np.maximum(
        mean_precision - std_precision,
        0
    ),
    np.minimum(
        mean_precision + std_precision,
        1
    ),
    color=ORANGE,
    alpha=0.16,
    linewidth=0,
    label="±1 SD across outer folds"
)

axes[1].axhline(
    y=y.mean(),
    linestyle="--",
    color=GRAY,
    linewidth=1,
    label=f"Positive prevalence = {y.mean():.2f}"
)

axes[1].set(
    xlabel="Recall",
    ylabel="Precision",
    title="b  Precision–recall curve"
)

axes[1].legend(
    loc="lower left"
)

for ax in axes:
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    sns.despine(ax=ax)

fig.suptitle(
    "Main analysis",
    fontsize=14,
    fontweight="bold"
)

fig.tight_layout()

save_figure(
    fig,
    OUTPUT_DIR,
    "main_binary_roc_pr_curves"
)

plt.show()
plt.close(fig)

true_all = predictions_df[
    "true_label"
].to_numpy()

predicted_all = predictions_df[
    "predicted_label"
].to_numpy()

cm_counts = confusion_matrix(
    true_all,
    predicted_all,
    labels=[0, 1]
)

cm_fraction = confusion_matrix(
    true_all,
    predicted_all,
    labels=[0, 1],
    normalize="true"
)

cm_annotations = np.empty_like(
    cm_fraction,
    dtype=object
)

for row in range(cm_fraction.shape[0]):
    for column in range(cm_fraction.shape[1]):
        cm_annotations[row, column] = (
            f"{cm_fraction[row, column] * 100:.1f}%"
            f"\n(n={cm_counts[row, column]})"
        )

pd.DataFrame(
    cm_counts,
    index=["true_0", "true_1"],
    columns=["predicted_0", "predicted_1"]
).to_csv(
    OUTPUT_DIR /
    "main_binary_confusion_matrix_counts.tsv",
    sep="\t"
)

fig, ax = plt.subplots(
    figsize=(5.4, 4.7)
)

sns.heatmap(
    cm_fraction * 100,
    annot=cm_annotations,
    fmt="",
    cmap=sns.light_palette(
        BLUE,
        as_cmap=True
    ),
    vmin=0,
    vmax=100,
    square=True,
    linewidths=1,
    linecolor="white",
    cbar_kws={
        "label": "Percentage within true class"
    },
    xticklabels=[
        "Strict filamentous",
        "Strict yeast"
    ],
    yticklabels=[
        "Strict filamentous",
        "Strict yeast"
    ],
    ax=ax
)

ax.set(
    xlabel="Predicted class",
    ylabel="True class",
    title="Main analysis"
)

plt.setp(
    ax.get_xticklabels(),
    rotation=20,
    ha="right"
)

plt.setp(
    ax.get_yticklabels(),
    rotation=0
)

fig.tight_layout()

save_figure(
    fig,
    OUTPUT_DIR,
    "main_binary_confusion_matrix"
)

plt.show()
plt.close(fig)

gain_summary = pd.DataFrame({
    "mean_gain": gain_df.mean(axis=1),
    "std_gain": gain_df.std(axis=1),
    "median_gain": gain_df.median(axis=1),
    "nonzero_fold_fraction": (
        gain_df > 0
    ).mean(axis=1),
})

gain_summary = gain_summary.sort_values(
    by=[
        "mean_gain",
        "nonzero_fold_fraction"
    ],
    ascending=[False, False],
    kind="mergesort"
)

gain_summary.index.name = "orthogroup"

gain_summary.to_csv(
    OUTPUT_DIR /
    "main_binary_xgbrf_gain_importance.tsv",
    sep="\t"
)

top_gain = (
    gain_summary
    .head(30)
    .sort_values(
        "mean_gain",
        ascending=True
    )
)

fig, ax = plt.subplots(
    figsize=(7.4, 11.5)
)

ax.barh(
    top_gain.index,
    top_gain["mean_gain"],
    color=BLUE,
    edgecolor="none",
    alpha=0.92,
    height=0.72
)

ax.set(
    xlabel="Mean gain importance across outer folds",
    ylabel="",
    title="Top 30 Orthogroups by XGBRF gain"
)

ax.grid(
    axis="x",
    linestyle="--",
    linewidth=0.6,
    alpha=0.35
)

sns.despine(ax=ax)
fig.tight_layout()

save_figure(
    fig,
    OUTPUT_DIR,
    "main_binary_top30_gain_importance"
)

plt.show()
plt.close(fig)

if RUN_SHAP and shap_df is not None:
    shap_summary = pd.DataFrame({
        "mean_abs_shap": shap_df.mean(axis=1),
        "std_abs_shap": shap_df.std(axis=1),
        "median_abs_shap": shap_df.median(axis=1),
        "nonzero_fold_fraction": (
            shap_df > 0
        ).mean(axis=1),
    })

    shap_summary = shap_summary.sort_values(
        by=[
            "mean_abs_shap",
            "nonzero_fold_fraction"
        ],
        ascending=[False, False],
        kind="mergesort"
    )

    shap_summary.index.name = "orthogroup"

    shap_summary.to_csv(
        OUTPUT_DIR /
        "main_binary_xgbrf_shap_importance.tsv",
        sep="\t"
    )

    top_shap = (
        shap_summary
        .head(30)
        .sort_values(
            "mean_abs_shap",
            ascending=True
        )
    )

    fig, ax = plt.subplots(
        figsize=(7.4, 11.5)
    )

    ax.barh(
        top_shap.index,
        top_shap["mean_abs_shap"],
        color=ORANGE,
        edgecolor="none",
        alpha=0.92,
        height=0.72
    )

    ax.set(
        xlabel="Mean absolute SHAP value across outer folds",
        ylabel="",
        title="Top 30 Orthogroups by SHAP importance"
    )

    ax.grid(
        axis="x",
        linestyle="--",
        linewidth=0.6,
        alpha=0.35
    )

    sns.despine(ax=ax)
    fig.tight_layout()

    save_figure(
        fig,
        OUTPUT_DIR,
        "main_binary_top30_shap_importance"
    )

    plt.show()
    plt.close(fig)

classification_report_df = pd.DataFrame(
    classification_report(
        true_all,
        predicted_all,
        labels=[0, 1],
        target_names=[
            "strict_filamentous",
            "strict_yeast"
        ],
        output_dict=True,
        zero_division=0
    )
).T

classification_report_df.to_csv(
    OUTPUT_DIR /
    "main_binary_classification_report.tsv",
    sep="\t"
)

print(f"Results saved to: {OUTPUT_DIR}")
